In [ ]:
import torch
import torch.nn as nn
import numpy as np
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(DoubleConv, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.conv(x)


class UNet(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(UNet, self).__init__()
        self.conv1 = DoubleConv(in_channels, 64)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.conv2 = DoubleConv(64, 128)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.conv3 = DoubleConv(128, 256)
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.conv4 = DoubleConv(256, 512)
        self.pool4 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.bottleneck = DoubleConv(512, 1024)

        self.upconv4 = nn.ConvTranspose2d(1024, 512, kernel_size=2, stride=2)
        self.iconv4 = DoubleConv(1024, 512)
        self.upconv3 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.iconv3 = DoubleConv(512, 256)
        self.upconv2 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.iconv2 = DoubleConv(256, 128)
        self.upconv1 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.iconv1 = DoubleConv(128, 64)

        self.outconv = nn.Conv2d(64, out_channels, kernel_size=1)

        self.softmax = nn.Softmax(dim=1)

    def forward(self, x):
        conv1 = self.conv1(x)
        pool1 = self.pool1(conv1)
        conv2 = self.conv2(pool1)
        pool2 = self.pool2(conv2)
        conv3 = self.conv3(pool2)
        pool3 = self.pool3(conv3)
        conv4 = self.conv4(pool3)
        pool4 = self.pool4(conv4)

        bottleneck = self.bottleneck(pool4)

        upconv4 = self.upconv4(bottleneck)
        cat4 = torch.cat((upconv4, conv4), dim=1)
        iconv4 = self.iconv4(cat4)
        upconv3 = self.upconv3(iconv4)
        cat3 = torch.cat((upconv3, conv3), dim=1)
        iconv3 = self.iconv3(cat3)
        upconv2 = self.upconv2(iconv3)
        cat2 = torch.cat((upconv2, conv2), dim=1)
        iconv2 = self.iconv2(cat2)
        upconv1 = self.upconv1(iconv2)
        cat1 = torch.cat((upconv1, conv1), dim=1)
        iconv1 = self.iconv1(cat1)

        out = self.outconv(iconv1)
        out = self.softmax(out)
        return out

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F



def conv(in_channels, out_channels):

    conv_block = nn.Sequential(
        nn.Conv2d(in_channels, out_channels, 3, padding=1),
        nn.BatchNorm2d(out_channels),
        nn.ReLU(inplace=True),
        nn.Conv2d(out_channels, out_channels, 3, padding=1),
        nn.BatchNorm2d(out_channels),
        nn.ReLU(inplace=True)
    )

    return conv_block


class BU_net1(nn.Module):
    def __init__(self, n_classes):
        super(BU_net1, self).__init__()

        self.convDown1 = conv(3, 64)
        self.convDown2 = conv(64, 128)
        self.convDown3 = conv(128, 256)
        self.convDown4 = conv(256, 512)
        self.convDown5 = conv(512, 1024)
        self.maxpool = nn.MaxPool2d(2, stride=2)
        self.convUp4 = conv(1024+512, 512)
        self.convUp3 = conv(512+256, 256)
        self.convUp2 = conv(256+128, 128)
        self.convUp1 = conv(128+64, 64)
        self.convUp_fin = nn.Conv2d(64, n_classes, kernel_size=1)

        self.upsample1 = nn.ConvTranspose2d(1024, 1024, kernel_size=32, stride=1)
        self.upsample2 = nn.ConvTranspose2d(512, 512, kernel_size=31, stride=1)
        self.upsample3 = nn.ConvTranspose2d(256, 256, kernel_size=61, stride=1)
        self.upsample4 = nn.ConvTranspose2d(128, 128, kernel_size=121, stride=1)


        self.sigmoid_layer = nn.Sigmoid()

    def forward(self, x):
        conv1 = self.convDown1(x)
        x = self.maxpool(conv1)
        conv2 = self.convDown2(x)
        x = self.maxpool(conv2)
        conv3 = self.convDown3(x)
        x = self.maxpool(conv3)
        conv4 = self.convDown4(x)
        x = self.maxpool(conv4)
        #WC_5 = self.WC(x)
        conv5 = self.convDown5(x)
        x = self.upsample1(conv5)

        x = F.interpolate(x, size=conv4.shape[2:4])  # conv4의 높이와 너비에 맞게 조절

        x = torch.cat([conv4,x], dim=1)
        x = self.convUp4(x)
        x = self.upsample2(x)

        x = F.interpolate(x, size=conv3.shape[2:4])
        x = torch.cat([conv3,x], dim=1)
        x = self.convUp3(x)
        x = self.upsample3(x)

        x = F.interpolate(x, size=conv2.shape[2:4])
        x = torch.cat([conv2,x], dim=1)
        x = self.convUp2(x)
        x = self.upsample4(x)

        x = F.interpolate(x, size=conv1.shape[2:4])
        x = torch.cat([conv1,x], dim=1)
        x = self.convUp1(x)
        out = self.convUp_fin(x)

        out = self.sigmoid_layer(out)

        return out

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F



class WC_Block(nn.Module):

    def __init__(self, in_channels, out_channels):
        super(WC_Block, self).__init__()

        self.split_conv_x1_1 = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=(15, 1)),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
            )
        self.split_conv_x1_2 = nn.Sequential(
            nn.Conv2d(out_channels, out_channels, kernel_size=(1, 15)),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
            )

        self.split_conv_x2_1 = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=(1, 15)),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
            )
        self.split_conv_x2_2 = nn.Sequential(
            nn.Conv2d(out_channels, out_channels, kernel_size=(15, 1)),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
            )
        self.conv_sum = nn.Conv2d(2* out_channels, out_channels, 3, padding=1)
        self.batch_norm = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
    def forward(self, x):

        split_conv_x1 = self.split_conv_x1_1(x)
        split_conv_x1 = self.split_conv_x1_2(split_conv_x1)
        split_conv_x2 = self.split_conv_x2_1(x)
        split_conv_x2 = self.split_conv_x2_2(split_conv_x2)
        x = torch.cat([split_conv_x1, split_conv_x2],dim=1)
        x = self.conv_sum(x)
        x = self.batch_norm(x)
        x = self.relu(x)

        return x


def conv(in_channels, out_channels):

    conv_block = nn.Sequential(
        nn.Conv2d(in_channels, out_channels, 3, padding=1),
        nn.BatchNorm2d(out_channels),
        nn.ReLU(inplace=True),
        nn.Conv2d(out_channels, out_channels, 3, padding=1),
        nn.BatchNorm2d(out_channels),
        nn.ReLU(inplace=True)
    )

    return conv_block


class BU_net2(nn.Module):
    def __init__(self, n_classes):
        super(BU_net2, self).__init__()

        self.convDown1 = conv(3, 64)
        self.convDown2 = conv(64, 128)
        self.convDown3 = conv(128, 256)
        self.convDown4 = conv(256, 512)
        self.convDown5 = nn.Sequential(
        nn.Conv2d(1024, 1024, 3, padding=1),
        nn.BatchNorm2d(1024),
        nn.ReLU(inplace=True)
        )
        self.maxpool = nn.MaxPool2d(2, stride=2)
        self.convUp4 = conv(1024+512, 512)
        self.convUp3 = conv(512+256, 256)
        self.convUp2 = conv(256+128, 128)
        self.convUp1 = conv(128+64, 64)
        self.convUp_fin = nn.Conv2d(64, n_classes, kernel_size=1)

        self.upsample1 = nn.ConvTranspose2d(1024, 1024, kernel_size=32, stride=1)
        self.upsample2 = nn.ConvTranspose2d(512, 512, kernel_size=31, stride=1)
        self.upsample3 = nn.ConvTranspose2d(256, 256, kernel_size=61, stride=1)
        self.upsample4 = nn.ConvTranspose2d(128, 128, kernel_size=121, stride=1)


        self.WC = WC_Block(512, 1024)

        self.sigmoid_layer = nn.Sigmoid()

    def forward(self, x):
        conv1 = self.convDown1(x)
        x = self.maxpool(conv1)
        conv2 = self.convDown2(x)
        x = self.maxpool(conv2)
        conv3 = self.convDown3(x)
        x = self.maxpool(conv3)
        conv4 = self.convDown4(x)
        x = self.maxpool(conv4)
        WC_5 = self.WC(x)
        conv5 = self.convDown5(WC_5)
        x = self.upsample1(conv5)

        x = F.interpolate(x, size=conv4.shape[2:4])  # conv4의 높이와 너비에 맞게 조절

        x = torch.cat([conv4,x], dim=1)
        x = self.convUp4(x)
        x = self.upsample2(x)

        x = F.interpolate(x, size=conv3.shape[2:4])
        x = torch.cat([conv3,x], dim=1)
        x = self.convUp3(x)
        x = self.upsample3(x)

        x = F.interpolate(x, size=conv2.shape[2:4])
        x = torch.cat([conv2,x], dim=1)
        x = self.convUp2(x)
        x = self.upsample4(x)

        x = F.interpolate(x, size=conv1.shape[2:4])
        x = torch.cat([conv1,x], dim=1)
        x = self.convUp1(x)
        out = self.convUp_fin(x)

        out = self.sigmoid_layer(out)

        return out

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class RES_Block(nn.Module):

    def __init__(self, in_channels, out_channels):
        super(RES_Block, self).__init__()

        self.split_conv_x1_1 = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=(15, 1), padding=(7, 0)),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
            )
        self.split_conv_x1_2 = nn.Sequential(
            nn.Conv2d(out_channels, out_channels, kernel_size=(1, 15), padding=(0, 7)),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
            )

        self.split_conv_x2_1 = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=(13, 1), padding=(6, 0)),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
            )
        self.split_conv_x2_2 = nn.Sequential(
            nn.Conv2d(out_channels, out_channels, kernel_size=(1, 13), padding=(0, 6)),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
            )

        self.split_conv_x3_1 = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=(11, 1), padding=(5, 0)),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
            )
        self.split_conv_x3_2 = nn.Sequential(
            nn.Conv2d(out_channels, out_channels, kernel_size=(1, 11),padding=(0, 5)),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
            )

        self.split_conv_x4_1 = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=(9, 1), padding=(4, 0)),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
            )
        self.split_conv_x4_2 = nn.Sequential(
            nn.Conv2d(out_channels, out_channels, kernel_size=(1, 9), padding=(0, 4)),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
            )

        self.sum_conv_x1 = nn.Sequential(
            nn.Conv2d(5 * out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
            )
        self.sum_conv_x2 = nn.Sequential(
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
            )
        self.sum_conv_x3 = nn.Sequential(
            nn.Conv2d(out_channels, out_channels, kernel_size=1, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
            )

    def forward(self, x):
        init = x

        split_conv_x1 = self.split_conv_x1_1(x)
        split_conv_x1 = self.split_conv_x1_2(split_conv_x1)

        split_conv_x2 = self.split_conv_x2_1(x)
        split_conv_x2 = self.split_conv_x2_2(split_conv_x2)

        split_conv_x3 = self.split_conv_x3_1(x)
        split_conv_x3 = self.split_conv_x3_2(split_conv_x3)

        split_conv_x4 = self.split_conv_x4_1(x)
        split_conv_x4 = self.split_conv_x4_2(split_conv_x4)


        x = torch.cat([init, split_conv_x1, split_conv_x2, split_conv_x3, split_conv_x4],dim=1)

        x = self.sum_conv_x1(x)
        x = self.sum_conv_x2(x)
        x = self.sum_conv_x3(x)

        return x


class WC_Block(nn.Module):

    def __init__(self, in_channels, out_channels):
        super(WC_Block, self).__init__()

        self.split_conv_x1_1 = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=(15, 1)),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
            )
        self.split_conv_x1_2 = nn.Sequential(
            nn.Conv2d(out_channels, out_channels, kernel_size=(1, 15)),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
            )

        self.split_conv_x2_1 = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=(1, 15)),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
            )
        self.split_conv_x2_2 = nn.Sequential(
            nn.Conv2d(out_channels, out_channels, kernel_size=(15, 1)),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
            )

        self.conv_sum = nn.Conv2d(2* out_channels, out_channels, 3, padding=1)
        self.batch_norm = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
    def forward(self, x):

        split_conv_x1 = self.split_conv_x1_1(x)
        split_conv_x1 = self.split_conv_x1_2(split_conv_x1)

        split_conv_x2 = self.split_conv_x2_1(x)
        split_conv_x2 = self.split_conv_x2_2(split_conv_x2)
        x = torch.cat([split_conv_x1, split_conv_x2],dim=1)

        x = self.conv_sum(x)
        x = self.batch_norm(x)
        x = self.relu(x)

        return x


def conv(in_channels, out_channels):

    conv_block = nn.Sequential(
        nn.Conv2d(in_channels, out_channels, 3, padding=1),
        nn.BatchNorm2d(out_channels),
        nn.ReLU(inplace=True),
        nn.Conv2d(out_channels, out_channels, 3, padding=1),
        nn.BatchNorm2d(out_channels),
        nn.ReLU(inplace=True)
    )

    return conv_block


class BU_net(nn.Module):
    def __init__(self, n_classes):
        super(BU_net, self).__init__()

        self.convDown1 = conv(3, 64)
        self.convDown2 = conv(64, 128)
        self.convDown3 = conv(128, 256)
        self.convDown4 = conv(256, 512)
        self.convDown5 = nn.Sequential(
        nn.Conv2d(1024, 1024, 3, padding=1),
        nn.BatchNorm2d(1024),
        nn.ReLU(inplace=True)
        )
        self.maxpool = nn.MaxPool2d(2, stride=2)
        self.convUp4 = conv(1024+512, 512)
        self.convUp3 = conv(512+256, 256)
        self.convUp2 = conv(256+128, 128)
        self.convUp1 = conv(128+64, 64)
        self.convUp_fin = nn.Conv2d(64, n_classes, kernel_size=1)

        self.upsample1 = nn.ConvTranspose2d(1024, 1024, kernel_size=32, stride=1)
        self.upsample2 = nn.ConvTranspose2d(512, 512, kernel_size=31, stride=1)
        self.upsample3 = nn.ConvTranspose2d(256, 256, kernel_size=61, stride=1)
        self.upsample4 = nn.ConvTranspose2d(128, 128, kernel_size=121, stride=1)

        self.RES1 = RES_Block(64, 64)
        self.RES2 = RES_Block(128, 128)
        self.RES3 = RES_Block(256, 256)
        self.RES4 = RES_Block(512, 512)
        self.WC = WC_Block(512, 1024)

        self.sigmoid_layer = nn.Sigmoid()

    def forward(self, x):
        conv1 = self.convDown1(x)
        x = self.maxpool(conv1)
        conv2 = self.convDown2(x)
        x = self.maxpool(conv2)
        conv3 = self.convDown3(x)
        x = self.maxpool(conv3)
        conv4 = self.convDown4(x)
        x = self.maxpool(conv4)
        WC_5 = self.WC(x)
        conv5 = self.convDown5(WC_5)
        x = self.upsample1(conv5)

        RES_4 = self.RES4(conv4)
        x = torch.cat([RES_4,x], dim=1)
        x = self.convUp4(x)
        x = self.upsample2(x)

        RES_3 = self.RES3(conv3)
        x = torch.cat([RES_3,x], dim=1)
        x = self.convUp3(x)
        x = self.upsample3(x)

        RES_2 = self.RES2(conv2)
        x = torch.cat([RES_2,x], dim=1)
        x = self.convUp2(x)
        x = self.upsample4(x)

        RES_1 = self.RES1(conv1)
        x = torch.cat([RES_1,x], dim=1)
        x = self.convUp1(x)
        out = self.convUp_fin(x)

        out = self.sigmoid_layer(out)

        return out

In [ ]:
import torch
import torch.nn as nn
import numpy as np

def get_model_summary(model, input_size):
    def get_num_params(module):
        return sum(p.numel() for p in module.parameters() if p.requires_grad)

    total_params = get_num_params(model)

    input_data = torch.randn(*input_size)
    forward_pass_size = 0
    params_size = total_params * 4 / (1024 ** 2)  # Size in MB (assuming 32-bit floats)

    def count_forward_hook(module, input, output):
        nonlocal forward_pass_size
        forward_pass_size += np.prod(output.size()) * 4 / (1024 ** 2)  # Size in MB

    hooks = []
    for layer in model.children():
        hooks.append(layer.register_forward_hook(count_forward_hook))

    with torch.no_grad():
        model(input_data)

    for hook in hooks:
        hook.remove()

    input_size_mb = np.prod(input_size) * 4 / (1024 ** 2)
    estimated_total_size = input_size_mb + forward_pass_size + params_size

    print("Total params: {:,}".format(total_params))
    print("Trainable params: {:,}".format(total_params))
    print("Non-trainable params: 0")
    print("Input size (MB): {:.2f}".format(input_size_mb))
    print("Forward/backward pass size (MB): {:.2f}".format(forward_pass_size))
    print("Params size (MB): {:.2f}".format(params_size))
    print("Estimated Total Size (MB): {:.2f}".format(estimated_total_size))

# Example usage:
class ExampleModel(nn.Module):
    def __init__(self):
        super(ExampleModel, self).__init__()
        self.layer1 = nn.Linear(10, 100)
        self.layer2 = nn.Linear(100, 1000)
        self.layer3 = nn.Linear(1000, 10)

    def forward(self, x):
        x = torch.sigmoid(self.layer1(x))
        x = torch.sigmoid(self.layer2(x))
        x = torch.sigmoid(self.layer3(x))
        return x

model = BU_net1(4)
input_size = (64, 3, 256, 256)
get_model_summary(model, input_size)